# How do I change the number of agents?

`set_agent_count(distribution_id, n)` rewrites the source so it spawns
`n` agents instead of whatever the scenario JSON specified.

In [ ]:
from datetime import datetime

print(f"Executed on {datetime.now().strftime('%d.%m.%Y, %H:%M')}")

from jupedsim_scenarios import load_scenario, run_scenario

scenario = load_scenario("../assets/bottleneck.zip")
scenario.set_agent_count(0, 30)   # 30 agents from the first distribution

result = run_scenario(scenario, seed=42)
print("agents:", result.total_agents)
print("evacuation time:", result.evacuation_time, "s")
result.cleanup()

Both forms work — by **index** or **string id**:

In [6]:
scenario.set_agent_count(0, 30)                       # index
scenario.set_agent_count("jps-distributions_0", 30)   # string id
# check 
print(scenario.distributions["jps-distributions_0"]["parameters"]["number"])

30


## Inspecting the raw scenario

The `Scenario` object exposes the parsed JSON as plain dicts. Use
`json.dumps(..., indent=2)` to pretty-print any sub-section.

In [3]:
import json

from jupedsim_scenarios import load_scenario

scenario = load_scenario("../assets/bottleneck.zip")

print(json.dumps(scenario.distributions, indent=2, default=str))

{
  "jps-distributions_0": {
    "type": "polygon",
    "coordinates": [
      [
        0,
        6
      ],
      [
        7.5,
        6
      ],
      [
        7.5,
        1.0000000000000001e-16
      ],
      [
        0,
        0
      ],
      [
        0,
        6
      ]
    ],
    "parameters": {
      "number": 50,
      "radius": 0.2,
      "v0": 1.3,
      "flow_start_time": 0,
      "flow_end_time": 10,
      "percentage": null,
      "distribution_mode": "by_number",
      "use_flow_spawning": false,
      "use_premovement": false,
      "premovement_distribution": "gamma",
      "premovement_param_a": null,
      "premovement_param_b": null,
      "premovement_seed": null,
      "radius_distribution": "constant",
      "v0_distribution": "constant"
    },
    "journey_weights": []
  }
}


Other dicts you can print the same way: `scenario.zones`, `scenario.stages`, `scenario.config`.

## Capacity check — why a "small" number can still be rejected

jupedsim-scenarios estimates how many agents fit in a distribution
area using a conservative packing approximation:

```
max_capacity ≈ floor( area / (π · r²) · 0.5 )
```

With the default radius of `0.2 m`, that's roughly **3 agents per
1 m²**. If `set_agent_count(i, n)` raises *"requested N agents but
area can hold at most ~M"*, either:

- reduce `n`, or
- enlarge the distribution polygon in the web editor, or
- shrink agent radius via `set_agent_params(i, radius=0.15)`.

You can compute a safe target from the current count instead of
hard-coding:

In [4]:
scenario = load_scenario("../assets/bottleneck.zip")
current = scenario.distributions["jps-distributions_0"]["parameters"]["number"]
print("current count:", current)

scenario.set_agent_count(0, current // 2)   # always within capacity

current count: 50
